In [ ]:
import os
import pandas as pd
# os.chdir() 改变 Python 进程的当前工作目录
# 加这一行是为了让后续的 'data/ShanghaiPM_Training.csv' 这个相对路径能够正确被找到
# 不管 Jupyter 是从哪个目录启动的，执行完这一行后，工作目录都会固定在项目根
os.chdir('D:/github/airpolprediction')
print('当前工作目录:', os.getcwd())

当前工作目录: D:\github\airpolprediction


In [ ]:
df_raw = pd.read_csv('data/ShanghaiPM_Training.csv', na_values='NA')
print('训练集形状:', df_raw.shape)
df_raw.info()

训练集形状: (52183, 17)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52183 entries, 0 to 52182
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   No             52183 non-null  int64  
 1   year           52183 non-null  int64  
 2   month          52183 non-null  int64  
 3   day            52183 non-null  int64  
 4   hour           52183 non-null  int64  
 5   season         52183 non-null  int64  
 6   PM_Jingan      24305 non-null  float64
 7   PM_US Post     33663 non-null  float64
 8   PM_Xuhui       24792 non-null  float64
 9   DEWP           52170 non-null  float64
 10  HUMI           52170 non-null  float64
 11  PRES           52155 non-null  float64
 12  TEMP           52170 non-null  float64
 13  cbwd           52171 non-null  object 
 14  Iws            52171 non-null  float64
 15  precipitation  48318 non-null  float64
 16  Iprec          48318 non-null  float64
dtypes: float64(10), int64(6), objec

In [ ]:
# pd.to_datetime() 的作用：
#   把包含日期/时间信息的列（或列的组合）转换成 Python 的 Datetime 类型
#   这里我们传入一个字典，告诉 pandas：
#     把 year 列当作"年"，month 列当作"月"，day 列当作"日"，hour 列当作"时"
df_raw['datetime'] = pd.to_datetime(
    {'year': df_raw['year'], 'month': df_raw['month'], 'day': df_raw['day'], 'hour': df_raw['hour']}
)

# df.set_index() 把指定的列变成 DataFrame 的行索引（类似 Excel 最左边那列行号）
df_raw.set_index('datetime', inplace=True)

# df.sort_index() 按行索引排序
df_raw.sort_index(inplace=True)

# df.drop() 删除不需要的列
# columns=[...] 指定要删除的列名列表
# errors='ignore' 表示如果某列不存在也不报错（容错处理）
df_raw.drop(columns=['No', 'year', 'month', 'day', 'hour'], inplace=True)

print('索引转换完成，当前列名:', list(df_raw.columns))
df_raw.head()

索引转换完成，当前列名: ['season', 'PM_Jingan', 'PM_US Post', 'PM_Xuhui', 'DEWP', 'HUMI', 'PRES', 'TEMP', 'cbwd', 'Iws', 'precipitation', 'Iprec']


,season,PM_Jingan,PM_US Post,PM_Xuhui,DEWP,HUMI,PRES,TEMP,cbwd,Iws,precipitation,Iprec
datetime,,,,,,,,,,,,
2010-01-01 00:00:00,4,NaN,NaN,NaN,-6.0,59.48,1026.1,1.0,cv,1.0,0.0,0.0
2010-01-01 01:00:00,4,NaN,NaN,NaN,-6.0,59.48,1025.1,1.0,SE,2.0,0.0,0.0
2010-01-01 02:00:00,4,NaN,NaN,NaN,-7.0,59.21,1025.1,0.0,SE,4.0,0.0,0.0
2010-01-01 03:00:00,4,NaN,NaN,NaN,-6.0,63.94,1024.0,0.0,SE,5.0,0.0,0.0
2010-01-01 04:00:00,4,NaN,NaN,NaN,-6.0,63.94,1023.0,0.0,SE,8.0,0.0,0.0


In [ ]:
KEEP_COLS = [
    'PM_Jingan',      # 静安站 PM2.5 浓度 (ug/m3)
    'PM_US Post',     # 美国领事馆站 PM2.5 浓度 (ug/m3)
    'PM_Xuhui',       # 徐汇站 PM2.5 浓度 (ug/m3)
    'TEMP',           # 气温 (摄氏度)
    'HUMI',           # 相对湿度 (%)
]

# df[[...]] 通过列名列表来筛选子集
# .copy() 创建一个独立的副本——这是好习惯，防止后续操作意外影响原始数据
df = df_raw[KEEP_COLS].copy()

print('最终数据形状:', df.shape)
print('列名:', list(df.columns))
print('索引类型:', type(df.index))  # 应该是 DatetimeIndex
df.head(10)

最终数据形状: (52183, 5)
列名: ['PM_Jingan', 'PM_US Post', 'PM_Xuhui', 'TEMP', 'HUMI']
索引类型: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>


,PM_Jingan,PM_US Post,PM_Xuhui,TEMP,HUMI
datetime,,,,,
2010-01-01 00:00:00,NaN,NaN,NaN,1.0,59.48
2010-01-01 01:00:00,NaN,NaN,NaN,1.0,59.48
2010-01-01 02:00:00,NaN,NaN,NaN,0.0,59.21
2010-01-01 03:00:00,NaN,NaN,NaN,0.0,63.94
2010-01-01 04:00:00,NaN,NaN,NaN,0.0,63.94
2010-01-01 05:00:00,NaN,NaN,NaN,0.0,59.21
2010-01-01 06:00:00,NaN,NaN,NaN,1.0,59.48
2010-01-01 07:00:00,NaN,NaN,NaN,1.0,64.18
2010-01-01 08:00:00,NaN,NaN,NaN,2.0,69.43


In [ ]:
# df.isna() 返回一个布尔型 DataFrame（True 表示该位置是缺失值 NaN）
# .sum() 对每列求和时，True 自动按 1 计数 → 得到每列的缺失数量
print('各列缺失数量:')
print(df.isna().sum())

# df.describe() 输出每一列的统计量：
#   count（非空个数）, mean（均值）, std（标准差）,
#   min（最小值）, 25%, 50%, 75%, max（最大值）
# .round(2) 把小数四舍五入到 2 位，让输出更整洁
print('\n统计摘要:')
df.describe().round(2)

各列缺失数量:
PM_Jingan     27878
PM_US Post    18520
PM_Xuhui      27391
TEMP             13
HUMI             13
dtype: int64

统计摘要:


,PM_Jingan,PM_US Post,PM_Xuhui,TEMP,HUMI
count,24305.00,33663.00,24792.00,52170.00,52170.00
mean,56.84,52.47,57.09,17.55,69.54
std,47.09,42.10,47.86,9.31,17.65
min,1.00,1.00,1.00,-5.00,11.32
25%,26.00,26.00,26.00,10.00,57.93
50%,43.00,41.00,43.00,19.00,72.42
75%,72.00,66.00,73.00,25.00,83.37
max,635.00,730.00,636.00,41.00,100.00



- **行索引**：DatetimeIndex（2010-01-01 ~ 2015-12-14 逐小时）
- **列**：PM_Jingan | PM_US Post | PM_Xuhui | TEMP | HUMI